In [ ]:
%pip install plotly
%pip install --upgrade nbformat

In [ ]:
import numpy as np
import plotly.graph_objects as go

size = 128
coords = np.linspace(-0.5, 0.5, size)
X, Y, Z = np.meshgrid(coords, coords, coords, indexing="ij")
#distance from center
R = np.sqrt(X**2 + Y**2)

#bottom blob 
bot = (R**2 / 0.40**2 + (Z + 0.15)**2 / 0.35**2) < 1.0

#top blob 
top = (R**2 / 0.28**2 + (Z - 0.20)**2 / 0.28**2) < 1.0

volume = (bot | top).astype(np.float32)

#shove some random cylinders through(perpendicular to Z)
rng = np.random.default_rng(69)
n_cylinders = 6
cyl_radius = 0.04

for _ in range(n_cylinders):
    z_pos = rng.uniform(-0.4, 0.4)
    angle = rng.uniform(0, np.pi)
    dx, dy = np.cos(angle), np.sin(angle)

    dist = np.sqrt((Y * 0 - (Z - z_pos) * dy)**2 +
                   ((Z - z_pos) * dx - X * 0)**2 +
                   (X * dy - Y * dx)**2)
    cyl = dist < cyl_radius
    volume[cyl & (volume > 0)] = 2.0

#bubbles
n_bubbles = 8
bubble_radius = 0.05

for _ in range(n_bubbles):
    cx = rng.uniform(-0.25, 0.25)
    cy = rng.uniform(-0.25, 0.25)
    cz = rng.uniform(-0.35, 0.35)
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2 + (Z - cz)**2)

    #maybe set material to 3.0 (if not air) , dont eat up rods
    volume[(dist < bubble_radius) & (volume != 2.0 )] = 0.0

#filled voxels
x, y, z = np.where(volume > 0)
vals = volume[x, y, z]
#plot every 5th point, get locations and materialvalues
xs, ys, zs, vs = x[::5], y[::5], z[::5], vals[::5]

#print(np.unique(volume))

fig = go.Figure(data=go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode='markers',
    marker=dict(
        size=1.5,
        color=vs,
        opacity=0.3,
    ),
))

fig.update_layout(
    scene=dict(
        xaxis=dict(showticklabels=False, title=''),
        yaxis=dict(showticklabels=False, title=''),
        zaxis=dict(showticklabels=False, title=''),
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    width=700, height=700,
)

fig.show()

Side Projection (Z is abcissa)

In [ ]:
projection = np.max(volume, axis=1)

import matplotlib.pyplot as plt

display = projection.copy()
plt.figure(figsize=(6, 6))
plt.imshow(display, origin='lower', aspect='auto')
plt.xlabel('Z')
plt.ylabel('X')
plt.title('Side projection')
plt.show()